# Multi-Case FEM Parameter and Position Sensitivity Stability Analysis (199 Cases)

This notebook examines the merged `FE_Results_Cases_All` pool: 199 retained cases numbered 0 through 200. Cases 25 and 30 were excluded during FEM generation.

The analysis aims to:

1. Analyse parameter and position associations using every element within each case.
2. Summarise the direction, magnitude and stability of these associations across 199 cases.
3. Identify variables consistently associated with maximum principal stress across operating conditions.
4. Examine whether high-stress regions follow consistent spatial patterns.
5. Inform the candidate feature pool for subsequent case-grouped baselines and symbolic regression.

Important qualifications:

- Each case contains 400,360 finite elements.
- The 199 cases contain 79,671,640 records, not that many independent FEM operating conditions.
- Machine-learning assignments must keep complete case_id groups together.
- The main analysis reads all elements one case at a time.
- Correlation describes statistical association, not an independent physical causal effect.

This is historical pre-split exploration. It is not the later development-only feature-selection procedure or a held-out final-test performance comparison.


## 0. Analysis Structure

Two levels of analysis are used.

**Single-case element-level analysis:** examine relationships between local parameter values, positions and maximum principal stress within each case.

**Multi-case stability analysis:** combine the within-case results to identify associations that recur across cases.

Rather than reporting only one pooled correlation, the notebook first produces records such as:

    case_id | feature | spearman | high_stress_difference | ...

It then summarises them across cases:

    feature | mean_abs_spearman | top3_count | direction_consistency | ...


In [ ]:
from pathlib import Path
import gc
import os
import re
import warnings

import numpy as np
import pandas as pd

# Keep matplotlib from trying to write caches into a non-writable home directory.
MPL_CACHE_DIR = Path("/tmp") / "matplotlib-nuclear-graphite-cache"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception as exc:
    MATPLOTLIB_AVAILABLE = False
    plt = None
    print("matplotlib unavailable; plotting cells will be skipped.")
    print(exc)

try:
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.ensemble import ExtraTreesRegressor
    from sklearn.inspection import permutation_importance
    SKLEARN_AVAILABLE = True
except Exception as exc:
    SKLEARN_AVAILABLE = False
    print("scikit-learn unavailable; optional model diagnostics will be skipped.")
    print(exc)

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    display = print

PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'src' / 'ct3_common.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory')

# The curated raw case directory is inside the project root.
CASE_DIR_OVERRIDE = PROJECT_ROOT / "FE_Results_Cases_All"
POSSIBLE_CASE_DIRS = [
    CASE_DIR_OVERRIDE,
    PROJECT_ROOT / "data" / "FE_Results_Cases_All",
]

if CASE_DIR_OVERRIDE is not None:
    CASE_DIR = Path(CASE_DIR_OVERRIDE)
else:
    existing_dirs = [path for path in POSSIBLE_CASE_DIRS if path.exists()]
    if not existing_dirs:
        raise FileNotFoundError(
            "Could not find FE_Results_Cases_All. Set CASE_DIR_OVERRIDE explicitly."
        )
    CASE_DIR = existing_dirs[0]

if not CASE_DIR.exists():
    raise FileNotFoundError(f"Case directory does not exist: {CASE_DIR}")

# Keep all 199-case outputs separate from historical 21-case results.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "multi_case_sensitivity_199_cases"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CASE_COUNT = 199
EXPECTED_CASE_NUMBER_MIN = 0
EXPECTED_CASE_NUMBER_MAX = 200
EXPECTED_MISSING_CASE_NUMBERS = {25, 30}

TARGET_COL = "sigma_max_principal"
PARAM_COLS = ["fluence_rate", "temperature", "weight_loss_rate"]
POSITION_COLS = [
    "rho",
    "rho_normalized",
    "theta",
    "theta_sin",
    "theta_cos",
    "z",
    "distance_to_inner_proxy",
    "distance_to_outer_proxy",
]
FEATURE_COLS = PARAM_COLS + POSITION_COLS

RAW_TO_STANDARD = {
    "ElementID": "element_id",
    "X": "x",
    "Y": "y",
    "Z": "z",
    "FluenceRate": "fluence_rate",
    "Temperature": "temperature",
    "WeightLossRate": "weight_loss_rate",
    "MaxPrincipalStress": "sigma_max_principal",
}

REQUIRED_RAW_COLS = list(RAW_TO_STANDARD.keys())
STANDARD_COLS = list(RAW_TO_STANDARD.values())

print("Project root:", PROJECT_ROOT)
print("Case dir:", CASE_DIR)
print("Output dir:", OUTPUT_DIR)

## 1. Discover Case Files and Check Structure

This step inspects filenames, headers, row counts and file sizes without parsing all numerical fields into data frames. Counting rows still scans the files.

- Confirm that all 199 retained cases are present.
- Check header consistency.
- Check element counts across cases.
- Build an inventory for subsequent case-by-case analysis.


In [ ]:
def extract_case_number(path: Path) -> int:
    match = re.search(r"Case_(\d+)", path.name)
    if not match:
        raise ValueError(f"Cannot extract case number from {path.name}")
    return int(match.group(1))


def count_data_rows(path: Path) -> int:
    count = 0
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            count += chunk.count(b"\n")
    return max(count - 1, 0)


case_files = sorted(CASE_DIR.glob("FE_Results_Case_*.txt"), key=extract_case_number)

case_numbers = [extract_case_number(path) for path in case_files]
number_span = set(range(EXPECTED_CASE_NUMBER_MIN, EXPECTED_CASE_NUMBER_MAX + 1))
missing_case_numbers = sorted(number_span - set(case_numbers))
unexpected_case_numbers = sorted(set(case_numbers) - number_span)
duplicate_case_numbers = sorted(
    number for number in set(case_numbers) if case_numbers.count(number) > 1
)
if not case_files:
    raise FileNotFoundError(f"No FE_Results_Case_*.txt files found in {CASE_DIR}")
if duplicate_case_numbers:
    raise ValueError(f"Duplicate case numbers found: {duplicate_case_numbers}")
if unexpected_case_numbers:
    raise ValueError(f"Case numbers outside the expected 0–200 range: {unexpected_case_numbers}")
if len(case_files) != EXPECTED_CASE_COUNT:
    print(
        f"WARNING: expected {EXPECTED_CASE_COUNT} cases but found {len(case_files)}. "
        "The notebook will continue and report the actual inventory."
    )
if set(missing_case_numbers) != EXPECTED_MISSING_CASE_NUMBERS:
    print(
        "WARNING: missing case numbers differ from the confirmed {25, 30}: "
        f"{missing_case_numbers}"
    )

inventory_records = []
for path in case_files:
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        header = handle.readline().strip()
    inventory_records.append({
        "case_id": f"case_{extract_case_number(path):02d}",
        "case_number": extract_case_number(path),
        "file_name": path.name,
        "file_path": str(path),
        "size_mb": path.stat().st_size / 1_000_000,
        "n_elements_from_line_count": count_data_rows(path),
        "header": header,
    })

case_inventory = pd.DataFrame(inventory_records)
inventory_path = OUTPUT_DIR / "case_file_inventory.csv"
case_inventory.to_csv(inventory_path, index=False)

print("Number of case files:", len(case_inventory))
print("Case-number range:", (min(case_numbers), max(case_numbers)))
print("Missing case numbers:", missing_case_numbers)
print("Total element rows from line counts:", int(case_inventory["n_elements_from_line_count"].sum()))
print("Saved:", inventory_path)
display(case_inventory)

if case_inventory["header"].nunique() != 1:
    print("WARNING: Not all headers are identical. Please inspect case_file_inventory.csv.")
if case_inventory["n_elements_from_line_count"].nunique() != 1:
    print("WARNING: Not all cases have the same number of elements.")

## 2. Reading, Column Standardisation and Derived Features

These functions provide shared preprocessing for each case:

- Strip whitespace from column names.
- Map the original FEM headers to standard code names.
- Derive cylindrical rho and theta coordinates.
- Derive theta_sin and theta_cos for periodic angular position.
- Derive rho_normalized for relative radial-position comparisons.
- Derive approximate distances to inner and outer radial bounds.

The radial distances are coordinate-based proxies, not verified distances to every physical surface.


In [ ]:
def read_case_file(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, skipinitialspace=True)
    raw.columns = [str(c).strip() for c in raw.columns]

    missing = [c for c in REQUIRED_RAW_COLS if c not in raw.columns]
    if missing:
        raise ValueError(f"{path.name} is missing expected columns: {missing}")

    df = raw[REQUIRED_RAW_COLS].rename(columns=RAW_TO_STANDARD).copy()
    for col in STANDARD_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    case_number = extract_case_number(path)
    df["case_id"] = f"case_{case_number:02d}"
    df["case_number"] = case_number
    df["source_file"] = path.name
    return df


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["rho"] = np.sqrt(out["x"] ** 2 + out["y"] ** 2)
    out["theta"] = np.arctan2(out["y"], out["x"])
    out["theta_sin"] = np.sin(out["theta"])
    out["theta_cos"] = np.cos(out["theta"])

    rho_min = out["rho"].min()
    rho_max = out["rho"].max()
    rho_span = rho_max - rho_min
    if rho_span == 0:
        out["rho_normalized"] = np.nan
    else:
        out["rho_normalized"] = (out["rho"] - rho_min) / rho_span

    out["distance_to_inner_proxy"] = out["rho"] - rho_min
    out["distance_to_outer_proxy"] = rho_max - out["rho"]

    # Scaled diagnostic feature. This is not a physical input for final formulas,
    # but can show whether the combined local parameter deviation relates to stress.
    scaled_terms = []
    for col in PARAM_COLS:
        mean = out[col].mean()
        std = out[col].std(ddof=0)
        centered = out[col] - mean
        out[f"{col}_centered"] = centered
        if std and not np.isnan(std):
            scaled_terms.append((centered / std) ** 2)
    if scaled_terms:
        out["param_field_magnitude_scaled"] = np.sqrt(np.sum(scaled_terms, axis=0))
    else:
        out["param_field_magnitude_scaled"] = np.nan

    return out


def validate_case_frame(df: pd.DataFrame, case_id: str) -> dict:
    return {
        "case_id": case_id,
        "n_elements": len(df),
        "element_id_unique": df["element_id"].nunique(dropna=True),
        "element_id_min": df["element_id"].min(),
        "element_id_max": df["element_id"].max(),
        "element_id_monotonic": bool(df["element_id"].is_monotonic_increasing),
        "missing_fraction_max": df[STANDARD_COLS].isna().mean().max(),
    }

## 3. Single-Case Analysis Functions

The following analyses are repeated for each complete case:

1. case_summary: stress, parameter and position statistics.
2. feature_correlations: Pearson and Spearman correlations with maximum principal stress.
3. high_stress_contrast: feature differences between the highest-stress 5% and the remaining elements.
4. binned_sensitivity: mean, p95 and maximum stress within feature bins.
5. location_hotspot_fraction: the fraction of high-stress elements within spatial bins.


In [ ]:
def percentile(series: pd.Series, q: float) -> float:
    return float(np.nanpercentile(series.to_numpy(), q))


def summarize_case(df: pd.DataFrame) -> dict:
    stress = df[TARGET_COL]
    record = {
        "case_id": df["case_id"].iloc[0],
        "case_number": int(df["case_number"].iloc[0]),
        "n_elements": len(df),
        "stress_mean": stress.mean(),
        "stress_std": stress.std(),
        "stress_min": stress.min(),
        "stress_p50": stress.quantile(0.50),
        "stress_p95": stress.quantile(0.95),
        "stress_p99": stress.quantile(0.99),
        "stress_max": stress.max(),
        "negative_stress_fraction": (stress < 0).mean(),
    }
    for col in PARAM_COLS + ["rho", "rho_normalized", "theta", "z"]:
        record[f"{col}_mean"] = df[col].mean()
        record[f"{col}_std"] = df[col].std()
        record[f"{col}_min"] = df[col].min()
        record[f"{col}_p95"] = df[col].quantile(0.95)
        record[f"{col}_max"] = df[col].max()
    return record


def compute_feature_correlations(df: pd.DataFrame, features: list[str], target: str = TARGET_COL) -> pd.DataFrame:
    records = []
    for feature in features:
        if feature not in df.columns:
            continue
        pair = df[[feature, target]].dropna()
        if len(pair) < 3 or pair[feature].nunique(dropna=True) < 2:
            continue
        pearson = pair[feature].corr(pair[target], method="pearson")
        spearman = pair[feature].corr(pair[target], method="spearman")
        records.append({
            "case_id": df["case_id"].iloc[0],
            "case_number": int(df["case_number"].iloc[0]),
            "feature": feature,
            "target": target,
            "pearson": pearson,
            "spearman": spearman,
            "abs_pearson": abs(pearson),
            "abs_spearman": abs(spearman),
            "n": len(pair),
        })
    out = pd.DataFrame(records)
    if not out.empty:
        out = out.sort_values(["case_number", "abs_spearman"], ascending=[True, False]).reset_index(drop=True)
    return out


def compute_high_stress_contrast(
    df: pd.DataFrame,
    features: list[str],
    target: str = TARGET_COL,
    quantile: float = 0.95,
) -> pd.DataFrame:
    threshold = df[target].quantile(quantile)
    high_mask = df[target] >= threshold
    high = df.loc[high_mask]
    rest = df.loc[~high_mask]
    records = []
    for feature in features:
        if feature not in df.columns:
            continue
        high_values = high[feature].dropna()
        rest_values = rest[feature].dropna()
        if len(high_values) < 2 or len(rest_values) < 2:
            continue
        pooled_std = np.sqrt((high_values.var(ddof=1) + rest_values.var(ddof=1)) / 2)
        diff = high_values.mean() - rest_values.mean()
        standardized = diff / pooled_std if pooled_std and not np.isnan(pooled_std) else np.nan
        records.append({
            "case_id": df["case_id"].iloc[0],
            "case_number": int(df["case_number"].iloc[0]),
            "feature": feature,
            "target": target,
            "high_stress_quantile": quantile,
            "threshold": threshold,
            "high_n": int(high_mask.sum()),
            "rest_n": int((~high_mask).sum()),
            "high_mean": high_values.mean(),
            "rest_mean": rest_values.mean(),
            "difference_high_minus_rest": diff,
            "standardized_difference": standardized,
            "abs_standardized_difference": abs(standardized),
            "high_median": high_values.median(),
            "rest_median": rest_values.median(),
        })
    out = pd.DataFrame(records)
    if not out.empty:
        out = out.sort_values(["case_number", "abs_standardized_difference"], ascending=[True, False]).reset_index(drop=True)
    return out


def compute_binned_sensitivity(
    df: pd.DataFrame,
    features: list[str],
    target: str = TARGET_COL,
    q: int = 10,
) -> pd.DataFrame:
    records = []
    for feature in features:
        if feature not in df.columns:
            continue
        tmp = df[[feature, target]].dropna().copy()
        if tmp[feature].nunique(dropna=True) < 3:
            continue
        try:
            tmp["_bin"] = pd.qcut(tmp[feature].rank(method="first"), q=q, labels=False, duplicates="drop")
        except ValueError:
            continue
        grouped = tmp.groupby("_bin", observed=True)
        for bin_id, group in grouped:
            records.append({
                "case_id": df["case_id"].iloc[0],
                "case_number": int(df["case_number"].iloc[0]),
                "feature": feature,
                "bin": int(bin_id),
                "n": len(group),
                "feature_min": group[feature].min(),
                "feature_max": group[feature].max(),
                "feature_mean": group[feature].mean(),
                "stress_mean": group[target].mean(),
                "stress_median": group[target].median(),
                "stress_p95": percentile(group[target], 95),
                "stress_max": group[target].max(),
            })
    return pd.DataFrame(records)


def compute_location_hotspot_fraction(
    df: pd.DataFrame,
    location_features: list[str] = ["rho_normalized", "theta", "z"],
    target: str = TARGET_COL,
    q: int = 12,
    high_quantile: float = 0.95,
) -> pd.DataFrame:
    threshold = df[target].quantile(high_quantile)
    records = []
    for feature in location_features:
        tmp = df[[feature, target]].dropna().copy()
        if tmp[feature].nunique(dropna=True) < 3:
            continue
        tmp["_is_high_stress"] = tmp[target] >= threshold
        try:
            tmp["_bin"] = pd.qcut(tmp[feature].rank(method="first"), q=q, labels=False, duplicates="drop")
        except ValueError:
            continue
        grouped = tmp.groupby("_bin", observed=True)
        for bin_id, group in grouped:
            records.append({
                "case_id": df["case_id"].iloc[0],
                "case_number": int(df["case_number"].iloc[0]),
                "feature": feature,
                "bin": int(bin_id),
                "n": len(group),
                "feature_min": group[feature].min(),
                "feature_max": group[feature].max(),
                "feature_mean": group[feature].mean(),
                "high_stress_threshold": threshold,
                "high_stress_fraction": group["_is_high_stress"].mean(),
                "stress_p95": percentile(group[target], 95),
                "stress_max": group[target].max(),
            })
    return pd.DataFrame(records)

## 4. Complete-Case Statistical Analysis

Read the 199 cases sequentially, retain the current case for calculation and release its data after summarisation.

The default analysis covers case summaries, feature correlations, high-stress contrasts, binned sensitivity, spatial hotspot fractions and coordinate-consistency checks.

This is a full-element analysis, with 400,360 elements per case. Runtime therefore includes parsing and processing the complete dataset, not just a plotting sample.


In [ ]:
case_summary_records = []
validation_records = []
correlation_frames = []
high_contrast_frames = []
binned_frames = []
hotspot_frames = []
coordinate_consistency_records = []

reference_coords = None
reference_case_id = None

for index, path in enumerate(case_files, start=1):
    print(f"[{index}/{len(case_files)}] Processing {path.name}")
    df = read_case_file(path)
    df = add_derived_features(df)
    case_id = df["case_id"].iloc[0]

    validation_records.append(validate_case_frame(df, case_id))
    case_summary_records.append(summarize_case(df))
    correlation_frames.append(
        compute_feature_correlations(df, FEATURE_COLS + ["param_field_magnitude_scaled"])
    )
    high_contrast_frames.append(
        compute_high_stress_contrast(df, FEATURE_COLS + ["param_field_magnitude_scaled"], quantile=0.95)
    )
    binned_frames.append(
        compute_binned_sensitivity(df, FEATURE_COLS, q=10)
    )
    hotspot_frames.append(
        compute_location_hotspot_fraction(df, location_features=["rho_normalized", "theta", "z"], q=12)
    )

    coords = df[["element_id", "x", "y", "z"]].copy()
    if reference_coords is None:
        reference_coords = coords
        reference_case_id = case_id
        coordinate_consistency_records.append({
            "case_id": case_id,
            "reference_case_id": reference_case_id,
            "same_element_id_sequence": True,
            "max_abs_dx": 0.0,
            "max_abs_dy": 0.0,
            "max_abs_dz": 0.0,
        })
    else:
        same_ids = bool(reference_coords["element_id"].equals(coords["element_id"]))
        max_abs = (reference_coords[["x", "y", "z"]] - coords[["x", "y", "z"]]).abs().max()
        coordinate_consistency_records.append({
            "case_id": case_id,
            "reference_case_id": reference_case_id,
            "same_element_id_sequence": same_ids,
            "max_abs_dx": max_abs["x"],
            "max_abs_dy": max_abs["y"],
            "max_abs_dz": max_abs["z"],
        })

    del df, coords
    gc.collect()

case_validation = pd.DataFrame(validation_records)
case_summary = pd.DataFrame(case_summary_records).sort_values("case_number").reset_index(drop=True)
feature_correlations = pd.concat(correlation_frames, ignore_index=True)
high_stress_contrast = pd.concat(high_contrast_frames, ignore_index=True)
binned_sensitivity = pd.concat(binned_frames, ignore_index=True)
location_hotspot_fraction = pd.concat(hotspot_frames, ignore_index=True)
coordinate_consistency = pd.DataFrame(coordinate_consistency_records).sort_values("case_id").reset_index(drop=True)

tables = {
    "case_validation": case_validation,
    "case_summary": case_summary,
    "feature_correlations_by_case": feature_correlations,
    "high_stress_contrast_by_case": high_stress_contrast,
    "binned_sensitivity_by_case": binned_sensitivity,
    "location_hotspot_fraction_by_case": location_hotspot_fraction,
    "coordinate_consistency": coordinate_consistency,
}

for name, table in tables.items():
    path = OUTPUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    print("Saved:", path)

display(case_summary.head())
display(feature_correlations.head(20))
display(high_stress_contrast.head(20))
display(coordinate_consistency.head())

## 5. Cross-Case Sensitivity Stability

Aggregate the per-case results at feature level. Key diagnostics include:

- mean_abs_spearman: average absolute monotonic association across cases.
- std_abs_spearman: variation in that association across cases.
- top3_count / top5_count: frequency of appearing in each case's top three or five features.
- direction_consistency: consistency of the correlation sign.
- mean_abs_high_stress_difference: average absolute contrast between high-stress and remaining regions.

A high mean association, low variability, frequent high ranks and consistent direction support considering a feature for further modelling. They do not establish causality or replace validation-based feature selection.


In [ ]:
correlation_ranked = feature_correlations.copy()
correlation_ranked["abs_spearman_rank"] = correlation_ranked.groupby("case_id")["abs_spearman"].rank(
    ascending=False, method="min"
)
correlation_ranked["abs_spearman_rank_score"] = correlation_ranked.groupby("case_id")["abs_spearman"].rank(
    pct=True
)

corr_stability = (
    correlation_ranked
    .groupby("feature")
    .agg(
        n_cases=("case_id", "nunique"),
        mean_abs_spearman=("abs_spearman", "mean"),
        median_abs_spearman=("abs_spearman", "median"),
        std_abs_spearman=("abs_spearman", "std"),
        max_abs_spearman=("abs_spearman", "max"),
        mean_spearman=("spearman", "mean"),
        positive_spearman_fraction=("spearman", lambda s: (s > 0).mean()),
        negative_spearman_fraction=("spearman", lambda s: (s < 0).mean()),
        top3_count=("abs_spearman_rank", lambda s: int((s <= 3).sum())),
        top5_count=("abs_spearman_rank", lambda s: int((s <= 5).sum())),
        mean_rank_score=("abs_spearman_rank_score", "mean"),
    )
    .reset_index()
)
corr_stability["direction_consistency"] = corr_stability[
    ["positive_spearman_fraction", "negative_spearman_fraction"]
].max(axis=1)
corr_stability = corr_stability.sort_values(
    ["mean_abs_spearman", "top5_count"], ascending=False
).reset_index(drop=True)

contrast_ranked = high_stress_contrast.copy()
contrast_ranked["high_stress_rank"] = contrast_ranked.groupby("case_id")["abs_standardized_difference"].rank(
    ascending=False, method="min"
)
contrast_ranked["high_stress_rank_score"] = contrast_ranked.groupby("case_id")["abs_standardized_difference"].rank(
    pct=True
)

contrast_stability = (
    contrast_ranked
    .groupby("feature")
    .agg(
        n_cases=("case_id", "nunique"),
        mean_abs_high_stress_difference=("abs_standardized_difference", "mean"),
        median_abs_high_stress_difference=("abs_standardized_difference", "median"),
        std_abs_high_stress_difference=("abs_standardized_difference", "std"),
        max_abs_high_stress_difference=("abs_standardized_difference", "max"),
        mean_standardized_difference=("standardized_difference", "mean"),
        positive_difference_fraction=("standardized_difference", lambda s: (s > 0).mean()),
        negative_difference_fraction=("standardized_difference", lambda s: (s < 0).mean()),
        top3_high_stress_count=("high_stress_rank", lambda s: int((s <= 3).sum())),
        top5_high_stress_count=("high_stress_rank", lambda s: int((s <= 5).sum())),
        mean_high_stress_rank_score=("high_stress_rank_score", "mean"),
    )
    .reset_index()
)
contrast_stability["high_stress_direction_consistency"] = contrast_stability[
    ["positive_difference_fraction", "negative_difference_fraction"]
].max(axis=1)
contrast_stability = contrast_stability.sort_values(
    ["mean_abs_high_stress_difference", "top5_high_stress_count"], ascending=False
).reset_index(drop=True)

corr_path = OUTPUT_DIR / "cross_case_correlation_stability.csv"
contrast_path = OUTPUT_DIR / "cross_case_high_stress_contrast_stability.csv"
corr_stability.to_csv(corr_path, index=False)
contrast_stability.to_csv(contrast_path, index=False)

print("Saved:", corr_path)
print("Saved:", contrast_path)
display(corr_stability)
display(contrast_stability)

## 6. Combined Sensitivity Ranking

The ranking combines two sources of evidence:

1. abs_spearman_rank_score: within-case ranking of absolute monotonic association.
2. high_stress_rank_score: within-case ranking of the feature contrast for the highest-stress 5%.

Model-based importance is not part of the default combined score. It requires additional fitting and case-grouped validation; an optional diagnostic is provided later.

A high combined score indicates relatively strong overall association and high-stress contrast across the analysed cases. It is an exploratory rank score, not a causal sensitivity derivative or a calibrated measure of predictive importance.


In [ ]:
case_level_rank_scores = (
    correlation_ranked[["case_id", "case_number", "feature", "abs_spearman", "spearman", "abs_spearman_rank_score"]]
    .merge(
        contrast_ranked[["case_id", "feature", "abs_standardized_difference", "standardized_difference", "high_stress_rank_score"]],
        on=["case_id", "feature"],
        how="outer",
    )
)
rank_score_cols = ["abs_spearman_rank_score", "high_stress_rank_score"]
case_level_rank_scores["combined_case_sensitivity_score"] = case_level_rank_scores[rank_score_cols].mean(axis=1)

combined_stability = (
    case_level_rank_scores
    .groupby("feature")
    .agg(
        n_cases=("case_id", "nunique"),
        mean_combined_score=("combined_case_sensitivity_score", "mean"),
        median_combined_score=("combined_case_sensitivity_score", "median"),
        std_combined_score=("combined_case_sensitivity_score", "std"),
        mean_abs_spearman=("abs_spearman", "mean"),
        mean_abs_high_stress_difference=("abs_standardized_difference", "mean"),
    )
    .reset_index()
    .sort_values("mean_combined_score", ascending=False)
    .reset_index(drop=True)
)

case_score_path = OUTPUT_DIR / "combined_case_level_rank_scores.csv"
combined_path = OUTPUT_DIR / "cross_case_combined_sensitivity_ranking.csv"
case_level_rank_scores.to_csv(case_score_path, index=False)
combined_stability.to_csv(combined_path, index=False)

print("Saved:", case_score_path)
print("Saved:", combined_path)
display(combined_stability)

## 7. Spatial Stability of High-Stress Regions

This analysis asks whether the highest-stress 5% of elements repeatedly concentrate in particular rho, theta or z intervals.

Such localisation can inform later damage or failure investigations, but this notebook does not calculate lifetime or establish a failure criterion. A stress map alone is insufficient for a lifetime prediction.


In [ ]:
hotspot_summary = (
    location_hotspot_fraction
    .groupby(["feature", "bin"])
    .agg(
        n_cases=("case_id", "nunique"),
        mean_feature_min=("feature_min", "mean"),
        mean_feature_max=("feature_max", "mean"),
        mean_feature_value=("feature_mean", "mean"),
        mean_high_stress_fraction=("high_stress_fraction", "mean"),
        median_high_stress_fraction=("high_stress_fraction", "median"),
        std_high_stress_fraction=("high_stress_fraction", "std"),
        mean_stress_p95=("stress_p95", "mean"),
        max_stress_max=("stress_max", "max"),
    )
    .reset_index()
    .sort_values(["feature", "mean_high_stress_fraction"], ascending=[True, False])
)

hotspot_path = OUTPUT_DIR / "cross_case_location_hotspot_stability.csv"
hotspot_summary.to_csv(hotspot_path, index=False)
print("Saved:", hotspot_path)

for feature in ["rho_normalized", "theta", "z"]:
    print(f"\nTop hotspot bins for {feature}:")
    display(hotspot_summary[hotspot_summary["feature"] == feature].head(8))

## 8. Case-Level Summary Analysis

Represent each case as one row and examine relationships between its overall parameter-field statistics and stress-distribution statistics.

The observational units here are the 199 FEM cases, not all element records. These are exploratory relationships across the supplied cases and should not be presented as definitive physical effects or held-out model-validation evidence.


In [ ]:
case_level_features = [
    "fluence_rate_mean",
    "fluence_rate_p95",
    "temperature_mean",
    "temperature_p95",
    "weight_loss_rate_mean",
    "weight_loss_rate_p95",
    "rho_normalized_mean",
    "theta_mean",
    "z_mean",
]
case_level_targets = ["stress_mean", "stress_p95", "stress_p99", "stress_max"]

case_level_records = []
for target in case_level_targets:
    for feature in case_level_features:
        pair = case_summary[[feature, target]].dropna()
        if len(pair) < 5 or pair[feature].nunique(dropna=True) < 2:
            continue
        case_level_records.append({
            "feature": feature,
            "target": target,
            "pearson": pair[feature].corr(pair[target], method="pearson"),
            "spearman": pair[feature].corr(pair[target], method="spearman"),
            "abs_spearman": abs(pair[feature].corr(pair[target], method="spearman")),
            "n_cases": len(pair),
        })

case_level_correlation = (
    pd.DataFrame(case_level_records)
    .sort_values(["target", "abs_spearman"], ascending=[True, False])
    .reset_index(drop=True)
)
case_level_corr_path = OUTPUT_DIR / "case_level_summary_correlations.csv"
case_level_correlation.to_csv(case_level_corr_path, index=False)
print("Saved:", case_level_corr_path)
display(case_level_correlation)

## 9. Optional Case-Grouped Model Diagnostic

This diagnostic is disabled by default. It rereads the case files and fits a model using sampled training elements. To run it in a separate experimental working copy, change:

    RUN_MODEL_DIAGNOSTIC = False

to:

    RUN_MODEL_DIAGNOSTIC = True

The diagnostic examines generalisation to cases excluded from each training fold; it does not replace symbolic regression. GroupKFold groups rows by case_id, preventing elements from the same case from entering both sides of a fold. This case-level safeguard alone does not implement the later similarity-group blocking policy.

This is a historical pre-split diagnostic, not the formal final-test protocol.


In [ ]:
RUN_MODEL_DIAGNOSTIC = False
MODEL_SAMPLE_PER_CASE = 5000
MODEL_RANDOM_SEED = 42


def sample_case_for_model(df: pd.DataFrame, n_per_case: int, random_seed: int = 42) -> pd.DataFrame:
    # Stratify by stress deciles so the sample keeps both ordinary and high-stress regions.
    tmp = df.copy()
    tmp["_stress_bin"] = pd.qcut(tmp[TARGET_COL].rank(method="first"), q=10, labels=False, duplicates="drop")
    sampled = (
        tmp.groupby("_stress_bin", group_keys=False, observed=True)
        .apply(lambda g: g.sample(
            n=max(1, min(len(g), n_per_case // max(1, tmp["_stress_bin"].nunique()))),
            random_state=random_seed,
        ))
    )
    high = tmp[tmp[TARGET_COL] >= tmp[TARGET_COL].quantile(0.95)]
    high_n = min(len(high), max(500, n_per_case // 5))
    if high_n > 0:
        high_sample = high.sample(n=high_n, random_state=random_seed)
        sampled = pd.concat([sampled, high_sample], ignore_index=True).drop_duplicates(
            subset=["case_id", "element_id"], keep="first"
        )
    return sampled.drop(columns=["_stress_bin"], errors="ignore")


if RUN_MODEL_DIAGNOSTIC:
    if not SKLEARN_AVAILABLE:
        raise ImportError("scikit-learn is required for model diagnostics.")

    sample_frames = []
    for index, path in enumerate(case_files, start=1):
        print(f"[{index}/{len(case_files)}] Sampling {path.name}")
        df = read_case_file(path)
        df = add_derived_features(df)
        sample_frames.append(sample_case_for_model(df, MODEL_SAMPLE_PER_CASE, MODEL_RANDOM_SEED))
        del df
        gc.collect()

    model_data = pd.concat(sample_frames, ignore_index=True)
    model_features = [
        "fluence_rate",
        "temperature",
        "weight_loss_rate",
        "rho_normalized",
        "theta_sin",
        "theta_cos",
        "z",
    ]
    model_data = model_data.dropna(subset=model_features + [TARGET_COL, "case_id"]).reset_index(drop=True)

    X = model_data[model_features]
    y = model_data[TARGET_COL]
    groups = model_data["case_id"]

    n_splits = min(5, model_data["case_id"].nunique())
    gkf = GroupKFold(n_splits=n_splits)

    fold_records = []
    importance_frames = []
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
        model = ExtraTreesRegressor(
            n_estimators=120,
            random_state=MODEL_RANDOM_SEED,
            n_jobs=-1,
            min_samples_leaf=5,
            max_features=0.8,
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X.iloc[test_idx])
        fold_records.append({
            "fold": fold,
            "train_rows": len(train_idx),
            "test_rows": len(test_idx),
            "train_cases": ",".join(sorted(groups.iloc[train_idx].unique())),
            "test_cases": ",".join(sorted(groups.iloc[test_idx].unique())),
            "mae": mean_absolute_error(y.iloc[test_idx], pred),
            "rmse": mean_squared_error(y.iloc[test_idx], pred, squared=False),
            "r2": r2_score(y.iloc[test_idx], pred),
        })

        perm = permutation_importance(
            model,
            X.iloc[test_idx],
            y.iloc[test_idx],
            n_repeats=5,
            random_state=MODEL_RANDOM_SEED,
            n_jobs=-1,
        )
        importance_frames.append(pd.DataFrame({
            "fold": fold,
            "feature": model_features,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }))

    model_metrics = pd.DataFrame(fold_records)
    model_importance = pd.concat(importance_frames, ignore_index=True)
    model_importance_summary = (
        model_importance
        .groupby("feature")
        .agg(
            mean_importance=("importance_mean", "mean"),
            std_importance=("importance_mean", "std"),
            max_importance=("importance_mean", "max"),
        )
        .reset_index()
        .sort_values("mean_importance", ascending=False)
    )

    model_metrics.to_csv(OUTPUT_DIR / "group_case_model_metrics.csv", index=False)
    model_importance.to_csv(OUTPUT_DIR / "group_case_permutation_importance_by_fold.csv", index=False)
    model_importance_summary.to_csv(OUTPUT_DIR / "group_case_permutation_importance_summary.csv", index=False)

    display(model_metrics)
    display(model_importance_summary)
else:
    print("RUN_MODEL_DIAGNOSTIC is False. Skipping optional group-by-case model diagnostic.")

## 10. Visualisations

The figures summarise:

1. Stress p95 and maximum stress across cases.
2. Average within-case Spearman associations across the case pool.
3. Stability of feature contrasts for high-stress regions.
4. Concentration of hotspots within rho, theta and z bins.


In [ ]:
if MATPLOTLIB_AVAILABLE:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(case_summary["case_id"], case_summary["stress_p95"], marker="o", label="stress p95")
    ax.plot(case_summary["case_id"], case_summary["stress_p99"], marker="o", label="stress p99")
    ax.plot(case_summary["case_id"], case_summary["stress_max"], marker="o", label="stress max")
    ax.set_title("Stress summary by case")
    ax.set_xlabel("case")
    ax.set_ylabel(TARGET_COL)
    ax.tick_params(axis="x", rotation=60)
    ax.legend()
    fig.tight_layout()
    fig_path = OUTPUT_DIR / "stress_summary_by_case.png"
    fig.savefig(fig_path, dpi=200)
    print("Saved:", fig_path)
    plt.show()

    top_corr = corr_stability.sort_values("mean_abs_spearman", ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top_corr["feature"], top_corr["mean_abs_spearman"])
    ax.set_title("Cross-case mean |Spearman correlation|")
    ax.set_xlabel("mean |Spearman| across cases")
    fig.tight_layout()
    fig_path = OUTPUT_DIR / "cross_case_mean_abs_spearman.png"
    fig.savefig(fig_path, dpi=200)
    print("Saved:", fig_path)
    plt.show()

    top_contrast = contrast_stability.sort_values("mean_abs_high_stress_difference", ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top_contrast["feature"], top_contrast["mean_abs_high_stress_difference"])
    ax.set_title("Cross-case mean high-stress standardized difference")
    ax.set_xlabel("mean absolute standardized difference")
    fig.tight_layout()
    fig_path = OUTPUT_DIR / "cross_case_high_stress_difference.png"
    fig.savefig(fig_path, dpi=200)
    print("Saved:", fig_path)
    plt.show()

    for feature in ["rho_normalized", "theta", "z"]:
        data = hotspot_summary[hotspot_summary["feature"] == feature].sort_values("bin")
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(data["mean_feature_value"], data["mean_high_stress_fraction"], marker="o")
        ax.set_title(f"Cross-case high-stress fraction by {feature}")
        ax.set_xlabel(feature)
        ax.set_ylabel("mean high-stress fraction")
        fig.tight_layout()
        fig_path = OUTPUT_DIR / f"cross_case_hotspot_fraction_{feature}.png"
        fig.savefig(fig_path, dpi=200)
        print("Saved:", fig_path)
        plt.show()
else:
    print("matplotlib unavailable; skipping plots.")

## 11. Interpreting the Outputs

| Output | Interpretation |
|---|---|
| case_summary.csv | Parameter, position and stress statistics for each case |
| feature_correlations_by_case.csv | Single-feature associations within each case |
| cross_case_correlation_stability.csv | Stability of associations across the 199 cases |
| high_stress_contrast_by_case.csv | Feature contrasts between the highest-stress 5% and remaining elements |
| cross_case_high_stress_contrast_stability.csv | Cross-case stability of high-stress contrasts |
| cross_case_combined_sensitivity_ranking.csv | Combined cross-case sensitivity ranks |
| cross_case_location_hotspot_stability.csv | Stability of spatial hotspot concentrations |
| case_level_summary_correlations.csv | Exploratory relationships between case-level parameter and stress summaries |

Use these outputs to distinguish widespread associations from case-specific ones, compare overall correlations with upper-tail contrasts, locate recurring spatial hotspots and propose features for subsequent validation.

The historical candidate pool proposed at this stage was:

    FluenceRate
    Temperature
    WeightLossRate
    rho_normalized
    theta_sin
    theta_cos
    Z

This historical proposal is not the final locked feature set. The later development-only ablation in notebook 00 selected the full periodic set using rho, rather than rho_normalized, alongside the three physical fields, theta_sin, theta_cos and z.


## 12. Multi-Case Distribution Plots

Two complementary types of distribution plots are provided:

1. **Case-level distributions:** one row per case, showing the distribution of summaries such as mean, p95, p99 and maximum stress across the 199 cases.
2. **Sampled element-level distributions:** a plotting sample from each case, showing stress, the three physical fields and spatial variables.

The complete dataset contains:

    199 cases * 400,360 elements = 79,671,640 rows (approximately 79.67 million)

Sampling here bounds plotting work and helps visualise distribution shapes and cross-case differences. It does not replace the preceding full-element statistics or reduce the population used for those statistics.


In [ ]:
RUN_DISTRIBUTION_PLOTS = False  # Run the core 199-case analysis first; enable later if needed.
DISTRIBUTION_SAMPLE_PER_CASE = 5000
DISTRIBUTION_RANDOM_SEED = 42

if RUN_DISTRIBUTION_PLOTS and MATPLOTLIB_AVAILABLE:
    # 1) Case-level distribution: one row per case.
    case_distribution_cols = [
        "stress_mean",
        "stress_p95",
        "stress_p99",
        "stress_max",
        "negative_stress_fraction",
    ]
    fig, axes = plt.subplots(1, len(case_distribution_cols), figsize=(4.2 * len(case_distribution_cols), 4))
    if len(case_distribution_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, case_distribution_cols):
        values = case_summary[col].dropna()
        ax.hist(values, bins=min(10, max(4, len(values) // 2)), edgecolor="black")
        ax.set_title(col)
        ax.set_xlabel(col)
        ax.set_ylabel("n_cases")
    fig.suptitle("Case-level stress metric distributions", y=1.03)
    fig.tight_layout()
    fig_path = OUTPUT_DIR / "case_level_stress_distribution_histograms.png"
    fig.savefig(fig_path, dpi=200, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()

    case_parameter_cols = [
        "fluence_rate_mean",
        "temperature_mean",
        "weight_loss_rate_mean",
        "fluence_rate_p95",
        "temperature_p95",
        "weight_loss_rate_p95",
    ]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.ravel()
    for ax, col in zip(axes, case_parameter_cols):
        values = case_summary[col].dropna()
        ax.hist(values, bins=min(10, max(4, len(values) // 2)), edgecolor="black")
        ax.set_title(col)
        ax.set_xlabel(col)
        ax.set_ylabel("n_cases")
    fig.suptitle("Case-level parameter distributions", y=1.02)
    fig.tight_layout()
    fig_path = OUTPUT_DIR / "case_level_parameter_distribution_histograms.png"
    fig.savefig(fig_path, dpi=200, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()

    # 2) Element-level sampled distribution: representative sample from each case.
    distribution_cols = [
        TARGET_COL,
        "fluence_rate",
        "temperature",
        "weight_loss_rate",
        "rho_normalized",
        "theta",
        "z",
    ]
    sample_frames = []
    for index, path in enumerate(case_files, start=1):
        print(f"[{index}/{len(case_files)}] Sampling for distribution plots: {path.name}")
        df = read_case_file(path)
        df = add_derived_features(df)
        sample_n = min(DISTRIBUTION_SAMPLE_PER_CASE, len(df))
        sample = df[["case_id", "case_number", "element_id"] + distribution_cols].sample(
            n=sample_n,
            random_state=DISTRIBUTION_RANDOM_SEED + extract_case_number(path),
        )
        sample_frames.append(sample)
        del df, sample
        gc.collect()

    element_distribution_sample = pd.concat(sample_frames, ignore_index=True)
    sample_path = OUTPUT_DIR / "element_distribution_sample.csv"
    element_distribution_sample.to_csv(sample_path, index=False)
    print("Saved:", sample_path)
    print("Sample shape:", element_distribution_sample.shape)

    sample_summary = (
        element_distribution_sample
        .groupby("case_id")[distribution_cols]
        .agg(["count", "mean", "std", "min", "median", "max"])
    )
    sample_summary.columns = [f"{col}_{stat}" for col, stat in sample_summary.columns]
    sample_summary = sample_summary.reset_index()
    sample_summary_path = OUTPUT_DIR / "element_distribution_sample_summary.csv"
    sample_summary.to_csv(sample_summary_path, index=False)
    print("Saved:", sample_summary_path)
    display(sample_summary.head())

    # Overall sampled histograms.
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.ravel()
    for ax, col in zip(axes, distribution_cols):
        values = element_distribution_sample[col].dropna()
        ax.hist(values, bins=60, edgecolor="black", alpha=0.85)
        ax.set_title(col)
        ax.set_xlabel(col)
        ax.set_ylabel("sampled_elements")
    for ax in axes[len(distribution_cols):]:
        ax.axis("off")
    fig.suptitle("Sampled element-level distributions across all cases", y=1.02)
    fig.tight_layout()
    fig_path = OUTPUT_DIR / "element_sample_distribution_histograms.png"
    fig.savefig(fig_path, dpi=200, bbox_inches="tight")
    print("Saved:", fig_path)
    plt.show()

    # Boxplots by case. These show whether distributions shift between cases.
    for col in distribution_cols:
        fig, ax = plt.subplots(figsize=(13, 4.8))
        element_distribution_sample.boxplot(
            column=col,
            by="case_id",
            ax=ax,
            grid=False,
            showfliers=False,
            rot=60,
        )
        ax.set_title(f"{col} distribution by case (sampled, outliers hidden)")
        ax.set_xlabel("case_id")
        ax.set_ylabel(col)
        fig.suptitle("")
        fig.tight_layout()
        fig_path = OUTPUT_DIR / f"element_sample_boxplot_by_case_{col}.png"
        fig.savefig(fig_path, dpi=200, bbox_inches="tight")
        print("Saved:", fig_path)
        plt.show()

elif RUN_DISTRIBUTION_PLOTS and not MATPLOTLIB_AVAILABLE:
    print("matplotlib unavailable; distribution plots skipped.")
else:
    print("RUN_DISTRIBUTION_PLOTS is False. Distribution plots skipped.")